Task A: PPO Alignment

Prasanna Paithankar (21CS30065)

Hugging Face Hub Links:
- Full Fine-tuning: PrasannaPaithankar/gpt2-medium-ppo-full
- Prefix Tuning: PrasannaPaithankar/gpt2-medium-ppo-prefix
- LoRA: PrasannaPaithankar/gpt2-medium-ppo-lora
- QLoRA: PrasannaPaithankar/gpt2-medium-ppo-qlora

In [1]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00


In [ ]:
import gc
import os

import torch
import torch.nn.functional as F
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, PrefixTuningConfig, TaskType, get_peft_model
from rich.progress import BarColumn, Progress, TaskProgressColumn, TextColumn
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

In [ ]:
ds = load_dataset("nrizwan/safe_ai_assignment_1")
ds["train"] = ds["train"].shuffle(seed=42).select(range(4000))

model_name = "gpt2-medium"
reward_model_name = "PrasannaPaithankar/bert-reward-model"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

reward_pipe = pipeline("text-classification", model=reward_model_name, device=0)


def collate_ppo(batch):
    return [item["Question"] for item in batch]


dataloader = DataLoader(ds["train"], batch_size=2, collate_fn=collate_ppo)

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

preference_train.csv:   0%|          | 0.00/81.5M [00:00<?, ?B/s]

preference_test.csv:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6000 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def setup(strategy):
    if strategy == "qlora":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32)
        return get_peft_model(model, peft_config)
    elif strategy == "lora":
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
        peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32)
        return get_peft_model(model, peft_config)
    elif strategy == "prefix":
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
        peft_config = PrefixTuningConfig(
            task_type=TaskType.CAUSAL_LM, num_virtual_tokens=20
        )
        return get_peft_model(model, peft_config)
    else:
        return AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto"
        )

In [ ]:
def batch_log_ps(logits, labels):
    shifted_logits = logits[..., :-1, :].contiguous()
    shifted_labels = labels[..., 1:].contiguous()
    loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
    token_logps = -loss_fct(
        shifted_logits.view(-1, shifted_logits.size(-1)), shifted_labels.view(-1)
    )
    token_logps = token_logps.view(shifted_labels.size())
    mask = shifted_labels != tokenizer.pad_token_id
    return (token_logps * mask).sum(-1)


def log_ps(model, prompts, responses, max_length=512):
    prompt_outputs = tokenizer(prompts, add_special_tokens=False)
    response_outputs = tokenizer(responses, add_special_tokens=False)

    prompt_ids_list = prompt_outputs["input_ids"]
    response_ids_list = response_outputs["input_ids"]

    batch_input_ids = []
    batch_labels = []

    max_len = 0
    for p_ids, r_ids in zip(prompt_ids_list, response_ids_list):
        seq = p_ids + r_ids
        label_seq = [tokenizer.pad_token_id] * len(p_ids) + r_ids

        if len(seq) > max_length:
            seq = seq[:max_length]
            label_seq = label_seq[:max_length]

        batch_input_ids.append(seq)
        batch_labels.append(label_seq)
        max_len = max(max_len, len(seq))

    padded_input_ids = []
    padded_labels = []
    attention_mask = []

    for seq, label_seq in zip(batch_input_ids, batch_labels):
        pad_len = max_len - len(seq)

        padded_input_ids.append(seq + [tokenizer.pad_token_id] * pad_len)
        padded_labels.append(label_seq + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append([1] * len(seq) + [0] * pad_len)

    input_ids_tensor = torch.tensor(
        padded_input_ids, dtype=torch.long, device=model.device
    )
    labels_tensor = torch.tensor(padded_labels, dtype=torch.long, device=model.device)
    attention_mask_tensor = torch.tensor(
        attention_mask, dtype=torch.long, device=model.device
    )

    with torch.amp.autocast("cuda", dtype=torch.float16):
        logits = model(
            input_ids=input_ids_tensor, attention_mask=attention_mask_tensor
        ).logits

    return batch_log_ps(logits, labels_tensor)

In [ ]:
def ppo_loop(strategy, epochs=1, ppo_epochs=2, kl_beta=0.1, clip_range=0.2):
    policy_model = setup(strategy)
    ref_model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    ).eval()
    for param in ref_model.parameters():
        param.requires_grad = False

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-5)
    step_rewards = []

    with Progress(
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TaskProgressColumn(),
    ) as progress:
        task = progress.add_task(
            f"[cyan]PPO {strategy.upper()}...", total=len(dataloader) * epochs
        )

        for epoch in range(epochs):
            for step, prompts in enumerate(dataloader):
                inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(
                    policy_model.device
                )
                with torch.no_grad():
                    outputs = policy_model.generate(
                        **inputs,
                        max_new_tokens=32,
                        do_sample=True,
                        top_p=0.9,
                        pad_token_id=tokenizer.eos_token_id,
                    )

                prompt_len = inputs.input_ids.shape[1]
                responses = tokenizer.batch_decode(
                    outputs[:, prompt_len:], skip_special_tokens=True
                )
                full_texts = [p + r for p, r in zip(prompts, responses)]

                # B. Get BERT Rewards
                bert_outputs = reward_pipe(full_texts)
                rewards = torch.tensor(
                    [out["score"] for out in bert_outputs],
                    device=policy_model.device,
                    dtype=torch.float16,
                )
                step_rewards.append(rewards.mean().item())

                with torch.no_grad():
                    ref_logps = log_ps(ref_model, prompts, responses)
                    old_pi_logps = log_ps(policy_model, prompts, responses)

                    kl_penalty = old_pi_logps - ref_logps
                    advantage = rewards - (kl_beta * kl_penalty)

                    if advantage.size(0) > 1:
                        advantage = advantage.to(torch.float32)
                        advantage = (advantage - advantage.mean()) / (
                            advantage.std() + 1e-5
                        )
                        advantage = advantage.to(torch.float16)

                for _ in range(ppo_epochs):
                    optimizer.zero_grad()
                    current_pi_logps = log_ps(policy_model, prompts, responses)

                    ratio = torch.exp(current_pi_logps - old_pi_logps)

                    surr1 = ratio * advantage
                    surr2 = (
                        torch.clamp(ratio, 1.0 - clip_range, 1.0 + clip_range)
                        * advantage
                    )
                    ppo_loss = -torch.min(surr1, surr2).mean()

                    ppo_loss.backward()

                    torch.nn.utils.clip_grad_norm_(
                        policy_model.parameters(), max_norm=1.0
                    )

                    optimizer.step()

                progress.update(
                    task,
                    advance=1,
                    description=f"[cyan]{strategy.upper()} | Reward: {rewards.mean().item():.4f} | Loss: {ppo_loss.item():.4f}",
                )

    policy_model.push_to_hub(f"PrasannPaithankar/gpt2-medium-ppo-{strategy}")

    del policy_model
    del ref_model
    del optimizer
    gc.collect()
    torch.cuda.empty_cache()

    return step_rewards

In [ ]:
all_ppo_rewards = {}

for strat in ["qlora", "lora", "prefix", "full"]:
    all_ppo_rewards[strat] = ppo_loop(strat)

import pickle

# Open a file in write-binary mode and dump the data
with open("losses.pkl", "wb") as file:
    pickle.dump(all_ppo_rewards, file)

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Output()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
